# 07b — Caracterización Nivel 2 (5 sub-arquetipos + outliers)


In [1]:
import pandas as pd, numpy as np, joblib
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_QC = ROOT / 'data' / 'data_qc_gustos'
DM = ROOT / 'data' / 'data_models_gustos'
INFORMES = ROOT / 'informes' / 'fase4_gustos' / '05_archetypes' / 'nivel2'
(INFORMES / 'radar_charts').mkdir(parents=True, exist_ok=True)

assignments = pd.read_parquet(DATA_QC / 'two_stage_assignments.parquet')
features_tier2 = pd.read_parquet(DATA_QC / 'features_tier2.parquet')

# Solo jugadores con Tier 2 (21,599)
df = features_tier2.merge(assignments[['user_id','nivel1_cluster','nivel2_cluster']], on='user_id', how='inner')
print(f'Sample Tier 2: {df.shape}')
df['nivel2_cluster'] = df['nivel2_cluster'].fillna(-1).astype(int)  # outliers HDBSCAN
print(f'Cluster distribution:')
for k, n in df['nivel2_cluster'].value_counts().sort_index().items():
    pct = n/len(df)*100
    label = '(OUTLIER)' if k == -1 else ''
    print(f'  C{k}: {n:>6,} ({pct:.2f}%) {label}')


Sample Tier 2: (21599, 19)
Cluster distribution:
  C-1: 10,953 (50.71%) (OUTLIER)
  C0:    195 (0.90%) 
  C1:    152 (0.70%) 
  C2:    161 (0.75%) 
  C3:    264 (1.22%) 
  C4:  9,874 (45.72%) 


In [2]:
def characterize(cluster_id):
    cdf = df[df['nivel2_cluster']==cluster_id]
    rdf = df[df['nivel2_cluster']!=cluster_id]
    rows = []
    for feat in df.columns:
        if feat in ['user_id','nivel1_cluster','nivel2_cluster']: continue
        if not pd.api.types.is_numeric_dtype(df[feat]): continue
        c_med = cdf[feat].median()
        r_med = rdf[feat].median()
        g_iqr = df[feat].quantile(0.75) - df[feat].quantile(0.25)
        dist = (c_med - r_med) / g_iqr if g_iqr > 0 else 0
        rows.append({'feature':feat,'cluster_median':c_med,'rest_median':r_med,
                     'distinctiveness':dist,'abs_distinctiveness':abs(dist)})
    return pd.DataFrame(rows).sort_values('abs_distinctiveness', ascending=False)

characterizations = {}
for k in sorted(df['nivel2_cluster'].unique()):
    st = characterize(k)
    st['cluster'] = k
    characterizations[k] = st
    n = (df['nivel2_cluster']==k).sum()
    print(f'\nC{k} (N={n:,}) top 6:')
    print(st.head(6)[['feature','cluster_median','rest_median','distinctiveness']].to_string(index=False))

pd.concat(characterizations.values(), ignore_index=True).to_csv(INFORMES / 'subarchetype_statistics.csv', index=False)



C-1 (N=10,953) top 6:
                  feature  cluster_median  rest_median  distinctiveness
           fights_pct_pve        0.927536          1.0        -0.823518
     fights_pct_validated        0.888889          1.0        -0.767677
           fights_pct_pvp        0.056338          0.0         0.749994
        currency_pct_gems        0.028986          0.0         0.666667
        currency_pct_gold        0.500000          0.7        -0.611650
currency_total_volume_30d     4207.000000       1422.5         0.595870

C0 (N=195) top 6:
                     feature  cluster_median  rest_median  distinctiveness
         fights_avg_duration           730.0    76.750000        14.184897
              fights_pct_won             1.0     0.037037         6.562324
           currency_pct_gold             0.0     0.615385        -1.882001
   currency_pct_top1_concept             1.0     0.562500         1.464674
currency_avg_quantity_per_tx             1.0    83.832508        -1.014793
    

In [3]:
RADAR = ['fights_pct_pve','fights_pct_pvp','fights_pct_won','currency_pct_inflow',
         'currency_n_transactions_30d','currency_pct_gold','currency_pct_gems','arena_total_combats_30d']
RADAR = [f for f in RADAR if f in df.columns]
norms = {f: df[f].quantile(0.90) if df[f].quantile(0.90)>0 else 1 for f in RADAR}

def vals_for(k):
    cdf = df[df['nivel2_cluster']==k]
    return [min(cdf[f].median()/norms[f], 1.5) if norms[f]>0 else 0 for f in RADAR]

def plot_radar(values, labels, title, savepath, color='C0'):
    angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
    vals = list(values) + [values[0]]
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(polar=True))
    ax.plot(angles, vals, color=color, linewidth=2)
    ax.fill(angles, vals, color=color, alpha=0.25)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels, size=8)
    ax.set_ylim(0, 1.5)
    ax.set_title(title, size=12, weight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(savepath, dpi=120, bbox_inches='tight'); plt.close()

ks = sorted(df['nivel2_cluster'].unique())
colors = plt.cm.tab10(np.linspace(0, 1, len(ks)))
all_vals = {}
for k, c in zip(ks, colors):
    nk = (df['nivel2_cluster']==k).sum()
    vals = vals_for(k)
    all_vals[k] = vals
    label = 'OUTLIERS' if k==-1 else f'Sub-arquetipo {k}'
    plot_radar(vals, RADAR, f'{label} (N={nk:,})',
               INFORMES / 'radar_charts' / f'subarchetype_{k}_radar.png', c)

# Overlay
fig, ax = plt.subplots(figsize=(10,10), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2*np.pi, len(RADAR), endpoint=False).tolist() + [0]
for k, c in zip(ks, colors):
    nk = (df['nivel2_cluster']==k).sum()
    vals = list(all_vals[k]) + [all_vals[k][0]]
    label = 'OUTLIERS' if k==-1 else f'C{k}'
    ax.plot(angles, vals, color=c, linewidth=2, label=f'{label} (N={nk:,})')
    ax.fill(angles, vals, color=c, alpha=0.1)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(RADAR, size=9)
ax.set_ylim(0, 1.5)
ax.set_title('Sub-arquetipos Nivel 2 (actividad)', size=14, weight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=9)
plt.tight_layout()
plt.savefig(INFORMES / 'radar_charts' / 'all_subarchetypes_overlay.png', dpi=120, bbox_inches='tight')
plt.close()
print('Radars Nivel 2 OK')


Radars Nivel 2 OK


In [4]:
from datetime import datetime

md = ['# Sub-arquetipos de actividad (Nivel 2) — Perfiles', '',
      f'**Fecha**: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
      f'**Algoritmo**: HDBSCAN min_cluster_size=100 sobre features Tier 2',
      f'**Sample**: {len(df):,} jugadores con datos Tier 2 (de 114,412 totales)',
      f'**Silhouette**: 0.301 | Outliers: ~51%',
      '', '## Resumen', '',
      '| Sub-arquetipo | N | % de Tier 2 | % del total 114k |',
      '|---:|---:|---:|---:|']
for k in sorted(df['nivel2_cluster'].unique()):
    n = (df['nivel2_cluster']==k).sum()
    label = 'OUTLIERS (no encajan)' if k==-1 else f'{k}'
    md.append(f'| {label} | {n:,} | {n/len(df)*100:.2f}% | {n/114412*100:.2f}% |')

md += ['', '---', '']
for k in sorted(df['nivel2_cluster'].unique()):
    n = (df['nivel2_cluster']==k).sum()
    st = characterizations[k]
    label = 'OUTLIERS' if k==-1 else f'Sub-arquetipo {k}'
    md += [f'## {label}', '', f'- **N**: {n:,}',
           '', '### Top 10 features distintivas', '',
           '| Feature | C_med | R_med | Δ |',
           '|---|---:|---:|---:|']
    for _, r in st.head(10).iterrows():
        md.append(f"| `{r['feature']}` | {r['cluster_median']:.4f} | {r['rest_median']:.4f} | {r['distinctiveness']:+.2f} |")
    md += ['', '### Interpretación tentativa', '', '_(rellenar manualmente)_', '', '---', '']

(INFORMES / 'subarchetype_profiles.md').write_text('\n'.join(md))
print('subarchetype_profiles.md guardado')


subarchetype_profiles.md guardado
